# EX04 — Investigation Walkthrough

**Evidence class legend**:
- 🟢 Deterministic keyless evidence
- 🔵 Representative deterministic fixture
- 🔴 Blocked (live provider required)

This notebook can be viewed without a paid provider. All live operations are clearly marked 🔴 Blocked.

## 1. Target Provenance

**Repository**: andela/buggy-python  
**Selected bug**: `IndexError` in `process_data` when `items` list is empty  
**Status**: 🔵 Fixture — live snapshot acquisition blocked (T7.01)  

See `artifacts/pre_fix/provenance.json` for full provenance record.

In [ ]:
import json
from pathlib import Path

ROOT = Path('..').resolve()
provenance = json.loads((ROOT / 'artifacts/pre_fix/provenance.json').read_text())
print(json.dumps(provenance, indent=2))

## 2. Graph and Vault Evidence (🔵 Fixture)

Live Graphify extraction is blocked (T7.01 — requires target repo and credentials).
The `obsidian/` vault contains representative fixture notes demonstrating the workflow.

In [ ]:
vault_path = ROOT / 'obsidian'
for f in sorted(vault_path.rglob('*.md')):
    print(f.relative_to(ROOT))

In [ ]:
print((vault_path / 'hot.md').read_text())

## 3. Graph-Analysis Extensions (🟢 Deterministic)

In [ ]:
import sys
sys.path.insert(0, str(ROOT / 'src'))

from ex04.shared.types import Entity, GraphData, Relationship
from ex04.services.analysis.orphan_detector import OrphanDetector
from ex04.services.analysis.patch_impact import PatchImpactAnalyzer

# Deterministic fixture graph matching the hot.md entities
graph = GraphData(
    entities=[
        Entity('BuggyModule', 'class', 'buggy.py', (1, 30)),
        Entity('process_data', 'function', 'buggy.py', (14, 28)),
        Entity('validate_input', 'function', 'buggy.py', (5, 12)),
        Entity('main', 'function', 'buggy.py', (32, 40)),
        Entity('helper', 'function', 'helper.py', (1, 10)),
    ],
    relationships=[
        Relationship('BuggyModule', 'process_data', 'contains'),
        Relationship('BuggyModule', 'validate_input', 'contains'),
        Relationship('process_data', 'validate_input', 'calls'),
        Relationship('main', 'process_data', 'calls'),
    ]
)

print(f'Entities: {len(graph.entities)}')
print(f'Relationships: {len(graph.relationships)}')

In [ ]:
# EXT-1: Orphan Detection
orphan_report = OrphanDetector().detect(graph, min_connections=0)
print(f'Orphan nodes: {[n.name for n in orphan_report.orphan_nodes]}')
print(f'Weak components: {orphan_report.weak_component_count}')
print(f'Limitations: {orphan_report.limitations}')

In [ ]:
# EXT-2: Patch Impact Analysis
impact_report = PatchImpactAnalyzer().analyze(graph, ['process_data'], max_depth=3)
print(f'Changed: {impact_report.changed_symbols}')
print(f'Direct dependents: {[n.name for n in impact_report.direct_dependents]}')
print(f'Transitive: {[n.name for n in impact_report.transitive_dependents]}')
print(f'Impact paths: {impact_report.impact_paths}')
print(f'Limitations: {impact_report.limitations}')

## 4. Run Evidence (🔵 Fixture)

In [ ]:
manifest = json.loads((ROOT / 'artifacts/manifests/fixture-001_manifest.json').read_text())
naive_manifest = json.loads((ROOT / 'artifacts/manifests/fixture-naive-001_manifest.json').read_text())

print('=== Graph-guided run ===')
print(f'Files read: {manifest["files_read"]}')
print(f'Source anchors: {manifest["source_anchors"]}')
print(f'Evidence class: {manifest["evidence_class"]}')

print()
print('=== Naive run ===')
print(f'Files read: {naive_manifest["files_read"]}')
print(f'Evidence class: {naive_manifest["evidence_class"]}')

print()
print('File read savings (fixture): '
      f'{naive_manifest["files_read"]} → {manifest["files_read"]} '
      f'({(1 - manifest["files_read"]/naive_manifest["files_read"])*100:.0f}% reduction)')

## 5. Correctness Gate (🔴 Blocked)

Live correctness gate requires the target repository and patch application.
Gate implementation is in `src/ex04/services/comparison/correctness_gate.py`.

**Gate verdict in fixture**: `skipped` (see `artifacts/runs/fixture-001/result.json`)

In [ ]:
result = json.loads((ROOT / 'artifacts/runs/fixture-001/result.json').read_text())
print(json.dumps(result['correctness_gate'], indent=2))

## 6. Known Limitations

| Operation | Status | Reason |
|---|---|---|
| Graphify extraction | 🔴 Blocked | Requires target repo and API key |
| Live LLM investigation | 🔴 Blocked | Requires provider credentials |
| Real token counts | 🔴 Blocked | No live provider runs |
| Correctness gate | 🔴 Blocked | Requires target repo |
| Graph-analysis extensions | 🟢 Complete | Deterministic, no API needed |
| Artifact structure | 🟢 Complete | Fixtures committed |
| SDK operations | 🟢 Complete | All exposed and tested |
| Test suite | 🟢 Complete | 412 tests, 97%+ coverage |